In [0]:
from pyspark.sql.types import StructType, StructField, StringType

columnas = [
    "Year", "Rank", "Country name", "Life evaluation (3-year average)",
    "Lower whisker", "Upper whisker", "Explained by: Log GDP per capita",
    "Explained by: Social support", "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices", "Explained by: Generosity",
    "Explained by: Perceptions of corruption", "Dystopia + residual",
]
schema = StructType([StructField(c, StringType(), True) for c in columnas])

df_whr_raw = (
    spark.read.format("excel")
    .option("headerRows", 1)
    .schema(schema)   # fuerza StringType por columna en la lectura misma, no corrige despues
    .load("/Volumes/happiness/landing/archivos/WHR/WHR26_Data_Figure_2.1.xlsx")
)
df_whr_raw.createOrReplaceTempView("whr_raw")
display(df_whr_raw)

In [0]:
%sql
CREATE OR REPLACE TABLE happiness.bronze.whr
COMMENT 'Bronze WHR (World Happiness Report 2026): panel objetivo de felicidad, una fila por pais-anio, con puntaje de vida y las 6 contribuciones precalculadas. 100% crudo, todas las columnas tal cual el archivo, sin castear ni filtrar (ver D10). Fuente: World Happiness Report + Gallup World Poll (ver D7: licencia no publicada, no se sube el xlsx crudo al repo).'
TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)
AS
SELECT
  *,
  current_timestamp() AS _ingest_ts,
  'WHR26_Data_Figure_2.1.xlsx' AS _source
FROM whr_raw;

SELECT COUNT(*) AS filas FROM happiness.bronze.whr;

In [0]:
%sql
select * from happiness.bronze.whr